# A2 — triangles + z-buffer

Barycentric depth interpolation + MSAA 2x2 bonus.

In [1]:
import numpy as np

def barycentric(x, y, v):
    x0, y0 = v[0][0], v[0][1]
    x1, y1 = v[1][0], v[1][1]
    x2, y2 = v[2][0], v[2][1]
    denom = (y1 - y2)*(x0 - x2) + (x2 - x1)*(y0 - y2)
    a = ((y1 - y2)*(x - x2) + (x2 - x1)*(y - y2)) / denom
    b = ((y2 - y0)*(x - x2) + (x0 - x2)*(y - y2)) / denom
    c = 1 - a - b
    return a, b, c

def inside(a, b, c):
    return a >= 0 and b >= 0 and c >= 0


In [2]:
def rasterize_triangle(t, W, H, framebuffer, zbuffer, color):
    v = [(p[0], p[1]) for p in t]
    z = np.array([p[2] for p in t])
    xs = [p[0] for p in v]; ys = [p[1] for p in v]
    x0, x1 = int(np.floor(min(xs))), int(np.ceil(max(xs)))
    y0, y1 = int(np.floor(min(ys))), int(np.ceil(max(ys)))
    for y in range(max(0, y0), min(H, y1)):
        for x in range(max(0, x0), min(W, x1)):
            a, b, c = barycentric(x + 0.5, y + 0.5, v)
            if not inside(a, b, c):
                continue
            zp = a*z[0] + b*z[1] + c*z[2]
            if zp < zbuffer[y, x]:
                zbuffer[y, x] = zp
                framebuffer[y, x] = color


## MSAA 2x2 (bonus)
Sample 4 sub-pixels per pixel. Naive version leaves a black seam on the shared edge because sub-samples that lose the z-test still contribute a background color. Fix: separate per-sample color and z buffers, resolve at the end.

In [3]:
def rasterize_msaa(t, W, H, sample_color, sample_z, color):
    v = [(p[0], p[1]) for p in t]
    z = np.array([p[2] for p in t])
    xs = [p[0] for p in v]; ys = [p[1] for p in v]
    x0, x1 = int(np.floor(min(xs))), int(np.ceil(max(xs)))
    y0, y1 = int(np.floor(min(ys))), int(np.ceil(max(ys)))
    offsets = [(0.25, 0.25), (0.75, 0.25), (0.25, 0.75), (0.75, 0.75)]
    for y in range(max(0, y0), min(H, y1)):
        for x in range(max(0, x0), min(W, x1)):
            for si, (dx, dy) in enumerate(offsets):
                a, b, c = barycentric(x + dx, y + dy, v)
                if not inside(a, b, c):
                    continue
                zp = a*z[0] + b*z[1] + c*z[2]
                if zp < sample_z[y, x, si]:
                    sample_z[y, x, si] = zp
                    sample_color[y, x, si] = color

def resolve(sample_color):
    return sample_color.mean(axis=2)
